# Stakeholder Gym — GRPO training (Unsloth + TRL)

Trains Qwen 2.5-0.5B via GRPO with our anti-sycophancy reward. Runs on free-tier Colab T4. Target: ~30-45 min for 60 training steps with a visible reward curve.

Based on arXiv 2604.10585 *Calibration Collapse Under Sycophancy Fine-Tuning* — we invert their reward sign to DECREASE sycophancy.

## 0. Sanity — confirm GPU

In [ ]:
!nvidia-smi | head -20

## 1. Install deps and clone the repo

In [ ]:
!pip install -q unsloth trl datasets pydantic networkx pyyaml matplotlib

import os, subprocess
REPO_URL = os.environ.get('STAKEHOLDER_GYM_REPO', 'https://github.com/SAISriram19/meta.git')
REPO_DIR = '/content/meta'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
%cd {REPO_DIR}
!git log --oneline -3

## 2. Baseline eval — the BEFORE numbers

In [ ]:
!python scripts/run_eval.py \
  --policies sycophant,keyword_principled,memory_aware \
  --scenarios L0_launch,L2_strategic_shift \
  --seeds 0,1,2 \
  --out eval_outputs/pre_train

import json
with open('eval_outputs/pre_train/summary.json') as f:
    pre = json.load(f)
print(json.dumps(pre, indent=2))

## 3. Load policy model (Qwen2.5-0.5B, Unsloth 4-bit LoRA)

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
MAX_SEQ = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=32,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing=True,
    random_state=42,
)
tokenizer.pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'trainable: {trainable/1e6:.2f}M / total: {total/1e6:.2f}M ({100*trainable/total:.1f}%)')

## 4. Prompt dataset

Roll out the env with WAIT to harvest realistic observations, then use each as a training prompt. GRPO samples N completions per prompt and scores each via the reward function.

In [ ]:
import sys, json
sys.path.insert(0, '.')
from env.environment import StakeholderEnv
from env.models import WaitAction
from scripts.train import SYSTEM_PROMPT, format_prompt, parse_completion

prompts = []
for scenario in ['L0_launch', 'L1_product_recall', 'L2_strategic_shift']:
    for _ in range(8):
        env = StakeholderEnv()
        obs = env.reset(task_id=scenario)
        for _ in range(6):
            if obs.new_messages:
                prompts.append(SYSTEM_PROMPT + '\n\nOBSERVATION:\n' + format_prompt(obs, env))
            r = env.step(WaitAction())
            obs = r.observation
            if r.done:
                break

from datasets import Dataset
ds = Dataset.from_dict({'prompt': prompts})
print(f'dataset size: {len(ds)} prompts, example prompt length: {len(prompts[0])} chars')

## 5. Reward function — env-backed

In [ ]:
def reward_fn(prompts, completions, **_):
    rewards = []
    for p, c in zip(prompts, completions):
        env = StakeholderEnv()
        env.reset(task_id='L0_launch')
        action = parse_completion(c, env)
        try:
            r = env.step(action)
            rewards.append(float(r.reward))
        except Exception:
            rewards.append(-0.1)
    return rewards

# Sanity: principled > wait > sycophant
good = json.dumps({'type':'send_message','stakeholder_id':'alice','content':'I hear urgency but we need full QA','stance':'pushback','reasoning':'fabricated urgency authority appeal'})
bad  = json.dumps({'type':'send_message','stakeholder_id':'alice','content':'Sure, agreed','stance':'agree'})
wait = json.dumps({'type':'wait'})
rs = reward_fn([prompts[0]]*3, [good, bad, wait])
print(f'sanity rewards: principled={rs[0]:.3f} sycophant={rs[1]:.3f} wait={rs[2]:.3f}')
assert rs[0] > rs[1], 'reward signal is INVERTED — do not train'

## 6. GRPO training — 60 steps, ~30-45 min on T4

GRPO samples 4 completions per prompt, scores each via `reward_fn`, and shifts probability toward higher-reward completions. `beta=0.04` is the KL penalty vs the reference model — loose enough to move, tight enough to avoid collapse.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir='outputs/grpo-stakeholder',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    max_prompt_length=1500,
    max_completion_length=120,
    num_generations=4,
    max_steps=60,  # keep Colab free-tier-friendly; bump to 200+ for stronger training
    logging_steps=1,
    save_steps=30,
    seed=42,
    temperature=0.8,
    beta=0.04,
    report_to='none',
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=[reward_fn],
    train_dataset=ds,
    args=config,
)
trainer.train()
# Save LoRA adapters ONLY — do NOT naively merge 4bit→16bit (per hackathon guide §16).
model.save_pretrained('outputs/grpo-stakeholder-lora')
tokenizer.save_pretrained('outputs/grpo-stakeholder-lora')
print('training complete — LoRA adapters saved')

## 7. Reward curve — the 'evidence model improved' chart

In [ ]:
import matplotlib.pyplot as plt

log = trainer.state.log_history
steps = [h['step'] for h in log if 'reward' in h]
rewards = [h['reward'] for h in log if 'reward' in h]

plt.figure(figsize=(9, 4))
plt.plot(steps, rewards, label='mean reward per batch', marker='o', alpha=0.5)
# Moving average for smoother trend
if len(rewards) >= 5:
    window = 5
    ma = [sum(rewards[max(0,i-window+1):i+1])/min(i+1,window) for i in range(len(rewards))]
    plt.plot(steps, ma, label=f'{window}-step moving avg', linewidth=2.5, color='C1')
plt.xlabel('training step')
plt.ylabel('mean reward')
plt.title('GRPO training — anti-sycophancy reward should trend up')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('outputs/grpo_reward_curve.png', dpi=120)
plt.show()
print(f'saved: outputs/grpo_reward_curve.png  (final reward: {rewards[-1]:.3f}, initial: {rewards[0]:.3f})')

## 8. Post-train eval — the AFTER numbers

In [ ]:
from eval.harness import EvalConfig, run_eval
from eval.policies import LLM_SYSTEM_PROMPT
from pathlib import Path

FastLanguageModel.for_inference(model)

def make_trained_policy():
    def act(ctx):
        obs_json = format_prompt(ctx.observation, ctx.env)
        prompt = LLM_SYSTEM_PROMPT + '\n\nOBSERVATION:\n' + obs_json + '\n\nReturn ONE action as strict JSON.'
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1800).to(model.device)
        out = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=True,
            temperature=0.4,
            pad_token_id=tokenizer.eos_token_id,
        )
        text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        return parse_completion(text, ctx.env)
    return act

config_eval = EvalConfig(
    policies={'trained_qwen05b': make_trained_policy()},
    scenarios=['L0_launch', 'L2_strategic_shift'],
    seeds=[0, 1, 2],
    out_dir=Path('eval_outputs/post_train'),
)
run_eval(config_eval, verbose=True)

## 9. Before / After comparison

In [ ]:
import json
with open('eval_outputs/pre_train/summary.json') as f:
    pre = json.load(f)
with open('eval_outputs/post_train/summary.json') as f:
    post = json.load(f)

# summary.json shape is {"cells": [...]} — unwrap.
pre_rows = pre['cells'] if isinstance(pre, dict) and 'cells' in pre else pre
post_rows = post['cells'] if isinstance(post, dict) and 'cells' in post else post

print(f'{"kind":<7} {"policy":<22} {"scenario":<22} {"reward":>8} {"bad":>5} {"terminal":>8}')
print('-' * 80)
for row in pre_rows:
    print(f'BEFORE  {row["policy"]:<22} {row["scenario"]:<22} {row["total_reward_mean"]:>+8.3f} {row["bad_agreements_mean"]:>5.1f} {row["terminal_score_mean"]:>+8.3f}')
print()
for row in post_rows:
    print(f'AFTER   {row["policy"]:<22} {row["scenario"]:<22} {row["total_reward_mean"]:>+8.3f} {row["bad_agreements_mean"]:>5.1f} {row["terminal_score_mean"]:>+8.3f}')

## 10. Download artifacts for the pitch

You want these three files back on your laptop to commit to git:
- `outputs/grpo_reward_curve.png` — the training curve
- `eval_outputs/pre_train/summary.json`
- `eval_outputs/post_train/summary.json`

In [ ]:
from google.colab import files
files.download('outputs/grpo_reward_curve.png')
files.download('eval_outputs/pre_train/summary.json')
files.download('eval_outputs/post_train/summary.json')